# 🧠 MAX-LLM LoRA Fine-Tuning Notebook (1-Click Google Colab)
Fine-tunes **Qwen2.5-Coder** on Maxwell's Master Agentic & AI Architecture dataset.
Uses **Unsloth** for 2x faster training and direct 4-bit GGUF export for Ollama.

> **Instructions:** Select **Runtime > Change runtime type > T4 GPU (Free)**, then click **Runtime > Run all**.

In [ ]:
# 1. Install Unsloth & Training Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft loralib sentencepiece bitsandbytes transformers datasets

In [ ]:
# 2. Upload master_dpo.json dataset
from google.colab import files
import os

if not os.path.exists('master_dpo.json'):
    print('Please upload master_dpo.json from .max/dataset/ on your computer:')
    uploaded = files.upload()
print('Dataset ready!')

In [ ]:
# 3. Load Qwen2.5-Coder Base Model with Unsloth 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
# Choose 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit' or '1.5B'
model_name = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    max_seq_length = max_seq_length,
)

In [ ]:
# 4. Train with Direct Preference Optimization (DPO)
import json
from datasets import Dataset
from trl import DPOTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

with open('master_dpo.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

dpo_dataset = Dataset.from_list(data)

training_args = TrainingArguments(
    output_dir = 'runs/max_output',
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs = 3,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 5,
    optim = 'adamw_8bit',
    seed = 3407,
)

trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = training_args,
    beta = 0.1,
    train_dataset = dpo_dataset,
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = max_seq_length // 2,
)

trainer.train()

In [ ]:
# 5. Export Directly to 4-bit Quantized GGUF for Ollama
model.save_pretrained_gguf('max_llm_gguf', tokenizer, quantization_method = 'q4_k_m')

# Download the GGUF file directly to your computer
import glob
from google.colab import files
gguf_files = glob.glob('max_llm_gguf/*.gguf')
if gguf_files:
    print(f'Downloading {gguf_files[0]}...')
    files.download(gguf_files[0])
else:
    print('Saved in max_llm_gguf directory!')